# SAM-WM frozen Kaggle benchmark
Run in order. Do not use the retired `SAM_WM_V41_KAGGLE_INPUT.zip` workflow. Final/OOD cells are held-out gates; do not run them until training/validation is frozen.

In [ ]:
!cd /kaggle/working && rm -rf SAM-WM && git clone https://github.com/AnnyaB/SAM-WM.git
%cd /kaggle/working/SAM-WM
!git rev-parse HEAD
!python -m pip install -q -e '.[dev]'
!make verify

## Train three seeds

In [ ]:
for seed in 0 1 2; do python train.py --seed $seed --out artifacts/freiburg; done

## Validation only — architecture/model selection may use these results

In [ ]:
for seed in 0 1 2; do python eval.py --checkpoint artifacts/freiburg/seed_${seed}/best.pt --data freiburg --split validation --out artifacts/eval/seed_${seed}; done

## Freeze checkpoint/config hashes
After this cell, do not alter preprocessing, architecture, hyperparameters, splits, or QC rules for the reported experiment.

In [ ]:
!git rev-parse HEAD
!sha256sum config/train.yaml artifacts/freiburg/seed_*/best.pt

## FINAL TEST — run once only after freeze

In [ ]:
for seed in 0 1 2; do python eval.py --checkpoint artifacts/freiburg/seed_${seed}/best.pt --data freiburg --split heldout --open-heldout --out artifacts/eval/seed_${seed}; done

## OOD-1 — Novi Sad, zero-shot, no recalibration

In [ ]:
for seed in 0 1 2; do python eval.py --checkpoint artifacts/freiburg/seed_${seed}/best.pt --data novisad --split heldout --open-heldout --out artifacts/eval/seed_${seed}; done

## OOD-2 — FAIRUrbTemp, zero-shot
Attach the official DOI 10.48620/93247 extracted dataset as a Kaggle Input. Choose the city before viewing SAM-WM results and use the same city for all seeds.

In [ ]:
FAIR_ROOT = '/kaggle/input/REPLACE_WITH_FAIRURBTEMP/extracted-root'
CITY = 'REPLACE_WITH_PREREGISTERED_CITY'
import subprocess
for seed in (0, 1, 2):
    subprocess.run([
        'python', 'eval.py', '--checkpoint', f'artifacts/freiburg/seed_{seed}/best.pt',
        '--data', 'fairurbtemp', '--root', FAIR_ROOT, '--city', CITY,
        '--split', 'heldout', '--open-heldout', '--out', f'artifacts/eval/seed_{seed}'
    ], check=True)

## Aggregate three-seed evidence

In [ ]:
!python summarize.py --root artifacts/eval --out artifacts/summary.json
from pathlib import Path
print(Path('artifacts/summary.json').read_text())